In [ ]:
import numpy as np
import pandas as pd

# Configuración de semilla para garantizar reproducibilidad en todas las ejecuciones
np.random.seed(42)

# Tamaño de la muestra (1000 hogares encuestados)
n_registros = 1000

# 1. Generación de variables demográficas y geográficas
codusu_list = [f"HOG_EPH_2026_{i:04d}" for i in range(1, n_registros + 1)]

# Regiones INDEC (1: GBA, 40: NOA, 41: NEA, 42: Cuyo, 43: Pampeana, 44: Patagonia)
regiones = np.random.choice(
  [1, 40, 41, 42, 43, 44],
  size=n_registros,
  p=[0.35, 0.15, 0.15, 0.10, 0.15, 0.10],
)

# Integrantes totales en el hogar (distribución Poisson centrada en 3 personas)
integrantes = np.random.poisson(lam=2.3, size=n_registros) + 1

# 2. Generación de variables socioeconómicas y educativas
# Nivel educativo del jefe/a de hogar (1: Primaria Inc, 2: Primaria Comp, 3: Secund Inc, 4:
#Secund Comp, 5: Sup/Univ Comp, 9: S/R)

nivel_ed = np.random.choice(
  [1, 2, 3, 4, 5, 9],
  size=n_registros,
  p=[0.08, 0.22, 0.30, 0.25, 0.13, 0.02],
)

# Ingreso Total Familiar (ITF): modelado log-normal condicionado por el nivel educativo
ingreso_base = {
  1: 280000,
  2: 420000,
  3: 580000,
  4: 850000,
  5: 1450000,
  9: 500000,
}
itf_list = []

for ed in nivel_ed:
  mu = ingreso_base[ed]
# Generar variación estocástica de ingresos
  val = np.random.lognormal(mean=np.log(mu), sigma=0.45)
  itf_list.append(round(val, 2))

itf_array = np.array(itf_list)

# Decil de Ingreso Total Familiar (1 a 10) calculado empíricamente
decif_array = pd.qcut(itf_array, q=10, labels=list(range(1, 11)))

# 3. Generación de variables tecnológicas (Brecha Digital)
# A mayor nivel socioeconómico/educativo, mayor probabilidad de poseer PC e Internet
prob_pc = []
prob_inet = []

for ed, decil in zip(nivel_ed, decif_array):
# Probabilidad base según educación
  p_pc = 0.25 + (ed * 0.11) if ed != 9 else 0.45
  p_inet = 0.45 + (ed * 0.08) if ed != 9 else 0.65

# Ajuste por decil de ingreso
  p_pc += (int(decil) - 5) * 0.03
  p_inet += (int(decil) - 5) * 0.025

  prob_pc.append(np.clip(p_pc, 0.10, 0.95))
  prob_inet.append(np.clip(p_inet, 0.25, 0.98))

# Generar tenencias (1: Sí, 2: No, 9: Sin Respuesta)
iv12_3 = [
np.random.choice([1, 2, 9], p=[p, 0.98 - p, 0.02]) for p in prob_pc
]
ii7 = [
np.random.choice([1, 2, 9], p=[p, 0.98 - p, 0.02]) for p in prob_inet
]

# 4. Estructuración del DataFrame con nomenclatura oficial INDEC

data = {
  "CODUSU": codusu_list,
  "REGION": regiones,
  "IX_TOT": integrantes,
  "NIVEL_ED": nivel_ed,
  "ITF": itf_array,
  "DECIFR": decif_array,
  "IV12_3": iv12_3, # ¿Posee computadora/laptop?
  "II7": ii7, # ¿Tiene conexión a internet fija?
}

df_eph = pd.DataFrame(data)

# 5. Inducir deliberadamente valores faltantes (NaN) para la práctica de limpieza de la Clase 2
mask_nulos_itf = np.random.choice(
[True, False], size=n_registros, p=[0.06, 0.94]
)
mask_nulos_pc = np.random.choice(
[True, False], size=n_registros, p=[0.03, 0.97]
)

df_eph.loc[mask_nulos_itf, "ITF"] = np.nan
df_eph.loc[mask_nulos_pc, "IV12_3"] = np.nan


# 6. Diagnóstico inicial del dataset

print("Dimensiones del dataset:")
print(df_eph.shape)

print("\nInformación general:")
df_eph.info()

print("\nValores faltantes por variable:")
print(df_eph.isna().sum())

print("\nValores de NIVEL_ED:")
print(df_eph["NIVEL_ED"].value_counts(dropna=False))

print("\nValores de IV12_3:")
print(df_eph["IV12_3"].value_counts(dropna=False))

print("\nValores de II7:")
print(df_eph["II7"].value_counts(dropna=False))


# 7. Exportar archivo CSV
nombre_archivo = "eph_tecnologia_microdatos.csv"
df_eph.to_csv(nombre_archivo, index=False)

print(
  f"Dataset generado exitosamente: '{nombre_archivo}' "
  f"con {df_eph.shape[0]} registros y"
  f"{df_eph.shape[1]} columnas."
)

# 8. Crear una copia para realizar la limpieza

df_clean = df_eph.copy()

# Reemplazar códigos de no respuesta por valores faltantes

df_clean["NIVEL_ED"] = df_clean["NIVEL_ED"].replace([9, -9, 99], np.nan)

df_clean["IV12_3"] = df_clean["IV12_3"].replace([9, -9, 99], np.nan)

df_clean["II7"] = df_clean["II7"].replace([9, -9, 99], np.nan)

df_clean["ITF"] = df_clean["ITF"].replace([-9, 99], np.nan)

df_clean["REGION"] = df_clean["REGION"].replace([-9, 99], np.nan)

# Verificar que no existan códigos de no respuesta pendientes

print("\nVerificación de códigos especiales:")

for columna in ["REGION", "IX_TOT", "NIVEL_ED", "ITF", "IV12_3", "II7"]:
    print(f"{columna}:")
    print(df_clean[columna].value_counts(dropna=False).sort_index())

# Recalcular deciles utilizando los ingresos disponibles

df_clean["DECIFR"] = pd.qcut(
    df_clean["ITF"],
    q=10,
    labels=range(1, 11)
)

# Comprobar los resultados

print("Valores faltantes después de la limpieza:")
print(df_clean.isna().sum())

print("\nDimensiones del dataset limpio:")
print(df_clean.shape)

print("\nInformación del dataset limpio:")
df_clean.info()

# 9. Exportar dataset limpio

nombre_archivo_limpio = "df_clean.csv"

df_clean.to_csv(nombre_archivo_limpio, index=False)

print("Archivo limpio generado correctamente.")
print("Nombre:", nombre_archivo_limpio)
print("Registros:", df_clean.shape[0])
print("Variables:", df_clean.shape[1])

print("VERIFICACIÓN FINAL")

print("Cantidad de hogares:", len(df_clean))
print("Cantidad de variables:", df_clean.shape[1])

print("¿Se conservaron los 1000 hogares?")
print(len(df_clean) == 1000)

print("¿Se conservaron las 8 variables?")
print(df_clean.shape[1] == 8)

print("¿Hay identificadores duplicados?")
print(df_clean["CODUSU"].duplicated().sum())


from google.colab import files
files.download(nombre_archivo_limpio)

Dimensiones del dataset:
(1000, 8)

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   CODUSU    1000 non-null   object  
 1   REGION    1000 non-null   int64   
 2   IX_TOT    1000 non-null   int64   
 3   NIVEL_ED  1000 non-null   int64   
 4   ITF       947 non-null    float64 
 5   DECIFR    1000 non-null   category
 6   IV12_3    964 non-null    float64 
 7   II7       1000 non-null   int64   
dtypes: category(1), float64(2), int64(4), object(1)
memory usage: 56.2+ KB

Valores faltantes por variable:
CODUSU       0
REGION       0
IX_TOT       0
NIVEL_ED     0
ITF         53
DECIFR       0
IV12_3      36
II7          0
dtype: int64

Valores de NIVEL_ED:
NIVEL_ED
3    318
4    232
2    217
5    134
1     82
9     17
Name: count, dtype: int64

Valores de IV12_3:
IV12_3
1.0    608
2.0    344
NaN     36
9.0     12
Name: count, d

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>